# Experiment 2 — Gemma 2 9B, swapping, calibration, and a small ensemble

**Run this locally on your RTX 5090**, in the same environment and project directory as your successful 2B training notebook. Restart the kernel first. This is **training + validation**, not the Kaggle hidden-test submission notebook.

The experiment trains **one new 9B model**. The existing 2B model is not retrained and its saved validation logits are reused. Default: BF16 LoRA, rank 32, 2,048 tokens, up to two epochs, early stopping, and a **16-hour training-phase limit**. Final evaluation/TTA, preparation, and model loading are additional; 20 hours total is a target, not a guaranteed bound. A preflight benchmark estimates the cost on your machine.

The original train/validation IDs are reused exactly. The old validation fold is further split into **selection** and **audit** subsets. Checkpoints, temperature, TTA, and blend weights are chosen on selection only; audit results are displayed afterward. The audit is not an untouched competition test set: the original 2B experiment already reported the full validation score.

Changes are a bundled stronger experiment, not an ablation proving which single change caused a gain. No external preference checkpoints, teacher models, synthetic labels, or label-leak datasets are used.

## 0. Settings

The completed 2B run path is already filled in. Keep that run directory: we need **`split.csv`, `valid_logits.npz`, and `run_summary.json`**, not only the notebook.

Use the existing installed packages. This notebook **does not upgrade PyTorch or install a new CUDA/quantization stack**. It uses ordinary BF16 LoRA, not QLoRA. A maximum-length backward test checks memory before training.

For a later interrupted-run resume, set `RESUME_CHECKPOINT` to the printed `checkpoints/checkpoint-...` directory and rerun in a fresh kernel. Keep all training settings unchanged. For re-evaluation only, set `EVALUATE_RUN` to a completed run directory; training is skipped and the saved best adapter is loaded. Do not set both.

In [1]:
from pathlib import Path

BASELINE_RUN = Path(r"C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_lora_2b_20261011_064837")
DATA_DIR = Path("./data")
OUTPUT_ROOT = Path("../tmp/outputs")
RESUME_CHECKPOINT = None   # Example: r"...\arena_9b_...\checkpoints\checkpoint-360"
EVALUATE_RUN = None        # Example: r"...\arena_9b_20261012_..."

MODEL_NAME = "google/gemma-2-9b-it"
SEED = 42
MAX_LENGTH = 2048
PROMPT_BUDGET = 384       # Only a cap when the COMPLETE input does not fit.
LORA_RANK = 32
LORA_ALPHA = 64
LEARNING_RATE = 5e-5
MAX_EPOCHS = 2
EFFECTIVE_BATCH_SIZE = 32
BATCH_CANDIDATES = [2, 4, 8]  # Benchmark and select a safe, fast physical batch.
MAX_TRAIN_HOURS = 16.0        # Includes periodic validation; excludes final evaluation.
ATTENTION_BACKEND = "eager"  # Preserves Gemma softcapping; no FlashAttention install needed.

# Try both orientations only during FINAL evaluation, not at every checkpoint.
RUN_SWAP_TTA = True
TEMPERATURES = [0.80, 0.90, 1.00, 1.10, 1.25, 1.50]
OLD_MODEL_WEIGHTS = [0.0, 0.1, 0.2, 0.3]  # 0 permits rejecting the 2B ensemble.
SELECTION_TOLERANCE = 0.0005  # Prefer cheaper recipes within this loss of the best.

In [2]:
import gc
import hashlib
import importlib.util
import inspect
import json
import math
import os
import re
import shutil
import sys
import time
from datetime import datetime
from importlib.metadata import version
from types import SimpleNamespace

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

needed = ["torch", "transformers", "peft", "accelerate", "datasets", "scipy", "sklearn", "safetensors"]
missing = [x for x in needed if importlib.util.find_spec(x) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}. Use the same kernel as your successful 2B run.")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from safetensors.torch import load_file
from scipy.special import softmax
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainerCallback, TrainingArguments, set_seed,
)
from tqdm.auto import tqdm
from IPython.display import display

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("This local training notebook needs your CUDA/BF16-enabled RTX 5090 environment.")
if int(os.environ.get("WORLD_SIZE", "1")) != 1:
    raise RuntimeError("This notebook is configured and loss-normalized for one GPU, not a distributed launch.")
DEVICE = torch.device("cuda:0")
assert "compute_loss_func" in inspect.signature(Trainer).parameters
assert not (RESUME_CHECKPOINT and EVALUATE_RUN)
assert MAX_LENGTH % 8 == 0 and MAX_EPOCHS >= 1 and MAX_TRAIN_HOURS > 0
assert all(EFFECTIVE_BATCH_SIZE % b == 0 for b in BATCH_CANDIDATES)
set_seed(SEED)
versions = {p: version(p) for p in ["torch", "transformers", "peft", "accelerate", "datasets", "scikit-learn"]}
print(json.dumps(versions, indent=2))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f"| free {free_bytes / 2**30:.1f} / {total_bytes / 2**30:.1f} GiB")
if free_bytes < 25 * 2**30:
    raise RuntimeError("Less than 25 GiB GPU memory is free. Restart the kernel and close other GPU model sessions first.")

# PEFT's optional TorchAO dispatcher can reject an OLD installed TorchAO even for
# a non-quantized model. Disable only that unused dispatcher check, before injection.
# This does NOT change Gemma computations or the saved weights.
import peft.tuners.lora.torchao as peft_torchao
try:
    peft_torchao.is_torchao_available()
except ImportError as exc:
    print("Disabling unused, incompatible TorchAO integration:", exc)
    peft_torchao.is_torchao_available = lambda: False

c:\Users\yana\Desktop\Library\AI\ai-competitions\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1012 06:50:08.080000 15884 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


{
  "torch": "2.12.1+cu130",
  "transformers": "5.19.0",
  "peft": "0.21.2",
  "accelerate": "1.15.0",
  "datasets": "5.1.0",
  "scikit-learn": "1.9.1"
}
NVIDIA GeForce RTX 5090 | free 30.3 / 31.8 GiB


## 1. Reuse the old split and check the old predictions

No resplitting of the outer train/validation fold: this is essential for a valid ensemble with the saved 2B predictions. All alignment is by **example ID**, never assumed row order. Changes to the training CSV or split fail clearly instead of silently creating leakage.

In [3]:
# The relative location supports the same project layout on another machine.
if not BASELINE_RUN.is_dir():
    alternate = OUTPUT_ROOT / "arena_lora_2b_20261011_064837"
    if alternate.is_dir():
        BASELINE_RUN = alternate
BASELINE_RUN = BASELINE_RUN.resolve()
for name in ["split.csv", "valid_logits.npz", "run_summary.json"]:
    if not (BASELINE_RUN / name).is_file():
        raise FileNotFoundError(f"Missing {BASELINE_RUN / name}. Set BASELINE_RUN to your completed 2B run.")
if not (DATA_DIR / "train.csv").is_file():
    raise FileNotFoundError("Run from the project directory containing data/train.csv, or edit DATA_DIR.")

LABEL_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]
LABEL_NAMES = ["win_a", "win_b", "tie"]
ID2LABEL = dict(enumerate(LABEL_NAMES))
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
old_summary = json.loads((BASELINE_RUN / "run_summary.json").read_text(encoding="utf-8"))
assert old_summary["config"]["model_name"] == "google/gemma-2-2b-it"
assert old_summary["config"]["label_names"] == LABEL_NAMES
raw = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
raw["id"] = raw["id"].astype(str)
assert raw["id"].is_unique
one_hot = raw[LABEL_COLUMNS].to_numpy()
assert np.isin(one_hot, [0, 1]).all() and (one_hot.sum(1) == 1).all()
raw["labels"] = one_hot.argmax(1).astype(np.int64)
split_manifest = pd.read_csv(BASELINE_RUN / "split.csv", dtype={"id": str})
assert split_manifest["id"].is_unique and set(split_manifest["split"]) == {"train", "valid"}
assert set(split_manifest["id"]) == set(raw["id"]), "Training IDs differ from the old split."
raw = raw.merge(split_manifest, on="id", how="left", validate="one_to_one", sort=False)

def align_logits(path, target_ids, target_labels):
    with np.load(path, allow_pickle=False) as saved:
        ids = saved["ids"].astype(str)
        logits = np.asarray(saved["logits"], dtype=np.float64)
        labels = np.asarray(saved["labels"], dtype=np.int64)
    index = pd.Index(ids)
    assert index.is_unique and set(ids) == set(map(str, target_ids)), "Saved prediction IDs do not match validation."
    order = index.get_indexer(list(map(str, target_ids)))
    assert (order >= 0).all() and logits.shape == (len(ids), 3)
    assert np.array_equal(labels[order], target_labels), "Saved labels or class order differ."
    assert np.isfinite(logits).all()
    return logits[order]

# Keep the original CSV order, then reorder saved 2B logits to that order.
train_frame = raw.loc[raw["split"].eq("train")].reset_index(drop=True).copy()
valid_frame = raw.loc[raw["split"].eq("valid")].reset_index(drop=True).copy()
y_valid = valid_frame["labels"].to_numpy(dtype=np.int64)
valid_ids = valid_frame["id"].to_numpy(dtype=str)
old_logits = align_logits(BASELINE_RUN / "valid_logits.npz", valid_ids, y_valid)
old_probabilities = softmax(old_logits, axis=-1)
old_loss = float(log_loss(y_valid, old_probabilities, labels=[0, 1, 2]))
assert abs(old_loss - old_summary["validation"]["valid_log_loss"]) < 1e-5, "Old validation score failed to reproduce."
print(f"Reproduced 2B validation log loss: {old_loss:.6f}")
print(f"Train: {len(train_frame):,} | original validation: {len(valid_frame):,}")

Reproduced 2B validation log loss: 0.964771
Train: 45,981 | original validation: 11,496


In [4]:
def parse_turns(value):
    turns = json.loads(value) if isinstance(value, str) else value
    if turns is None:
        return [""]
    if not isinstance(turns, list) or not all(t is None or isinstance(t, str) for t in turns):
        raise ValueError("Each competition text field must decode to a list of strings/nulls.")
    return ["" if t is None else t for t in turns] or [""]


def format_turns(turns):
    return "\n\n".join(f"[Turn {i + 1}]\n{t if t else '[No text]'}" for i, t in enumerate(turns))


def prepare_frame(frame):
    records = []
    for row in frame.itertuples(index=False):
        try:
            p, a, b = [parse_turns(v) for v in (row.prompt, row.response_a, row.response_b)]
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Bad JSON in example {row.id}: {exc}") from exc
        normalized = json.dumps([" ".join(t.split()) for t in p], ensure_ascii=False)
        group = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
        n = max(map(len, [p, a, b]))
        p, a, b = [ts + [""] * (n - len(ts)) for ts in [p, a, b]]
        records.append({"id": str(row.id), "prompt_text": format_turns(p),
                        "response_a_text": format_turns(a), "response_b_text": format_turns(b),
                        "prompt_group": group, "num_turns": n, "labels": int(row.labels)})
    return pd.DataFrame(records)

train_frame = prepare_frame(train_frame)
valid_frame = prepare_frame(valid_frame)
assert not set(train_frame.prompt_group).intersection(valid_frame.prompt_group), "Prompt leakage across the old split."
inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED + 100)
selection_idx, audit_idx = next(inner.split(np.zeros(len(y_valid)), y_valid, valid_frame.prompt_group))
assert not set(valid_frame.iloc[selection_idx].prompt_group).intersection(valid_frame.iloc[audit_idx].prompt_group)
assert len(selection_idx) + len(audit_idx) == len(y_valid)
print(f"Selection: {len(selection_idx):,} | audit: {len(audit_idx):,} | shared prompt groups: 0")

# Stable per-ID swaps. One orientation per example: no doubling of training rows.
def should_swap(example_id):
    digest = hashlib.sha256(f"{SEED}:swap:{example_id}".encode()).digest()
    return digest[0] < 128

swap_mask = train_frame["id"].map(should_swap).to_numpy(dtype=bool)
train_frame.loc[swap_mask, ["response_a_text", "response_b_text"]] = (
    train_frame.loc[swap_mask, ["response_b_text", "response_a_text"]].to_numpy()
)
train_frame.loc[swap_mask, "labels"] = np.asarray([1, 0, 2])[train_frame.loc[swap_mask, "labels"].to_numpy()]
print(f"Training examples swapped: {swap_mask.mean():.1%}. Tie stays class 2.")

Selection: 5,747 | audit: 5,749 | shared prompt groups: 0
Training examples swapped: 50.2%. Tie stays class 2.


In [5]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

experiment = {
    "model_name": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
    "prompt_budget": PROMPT_BUDGET, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE, "max_epochs": MAX_EPOCHS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE, "attention_backend": ATTENTION_BACKEND,
    "method": "BF16 LoRA; no quantization", "preprocessing_version": "full-if-fits-v2",
    "augmentation": "deterministic per-ID 50% A/B swap; same orientation each epoch",
    "label_names": LABEL_NAMES, "baseline_run": str(BASELINE_RUN),
    "split_sha256": sha256_file(BASELINE_RUN / "split.csv"),
    "training_csv_sha256": sha256_file(DATA_DIR / "train.csv"),
}
previous_config = None
if EVALUATE_RUN:
    RUN_DIR = Path(EVALUATE_RUN).resolve()
elif RESUME_CHECKPOINT:
    RESUME_CHECKPOINT = str(Path(RESUME_CHECKPOINT).resolve())
    RUN_DIR = Path(RESUME_CHECKPOINT).parent.parent
else:
    RUN_DIR = OUTPUT_ROOT / f"arena_9b_{datetime.now():%Y%m%d_%H%M%S}"
    RUN_DIR.mkdir(parents=True, exist_ok=False)
RUN_DIR = RUN_DIR.resolve()
if EVALUATE_RUN or RESUME_CHECKPOINT:
    previous_config = json.loads((RUN_DIR / "run_config.json").read_text(encoding="utf-8"))
    if RESUME_CHECKPOINT and previous_config.get("versions") != versions:
        raise RuntimeError("Package versions changed. Resume in the original training environment.")
    for key, value in experiment.items():
        if key != "baseline_run" and previous_config.get(key) != value:
            raise ValueError(f"Run setting changed: {key}. Restore the saved setting before resuming/re-evaluating.")
else:
    shutil.copy2(BASELINE_RUN / "split.csv", RUN_DIR / "split.csv")

inner_manifest = valid_frame[["id", "prompt_group"]].copy()
inner_manifest["subset"] = "audit"
inner_manifest.loc[selection_idx, "subset"] = "selection"
if previous_config:
    previous_inner = pd.read_csv(RUN_DIR / "validation_subsets.csv", dtype={"id": str})
    assert previous_inner.equals(inner_manifest), "Validation subset reconstruction changed."
else:
    inner_manifest.to_csv(RUN_DIR / "validation_subsets.csv", index=False)
print("Run directory:", RUN_DIR)

Run directory: C:\Users\yana\Desktop\Library\AI\ai-competitions\tmp\outputs\arena_9b_20261012_065015


## 2. Tokenization: keep all text when it fits; otherwise protect both answers

The old model's **cached** logits retain its original 1,536-token preprocessing. Only the new 9B model uses the new preprocessing below.

The overall limit includes every header, BOS, and EOS. When over budget, cap the prompt, share the answer budget, reuse spare tokens, and preserve the beginning/end of each truncated section. This can still omit middle turns; the audit quantifies how often truncation occurs. PAB layout and turn numbering are retained rather than introducing a complicated template.

In [6]:
# Resolve an exact commit from one cached/downloaded config file, NOT a full snapshot.
# This avoids the IncompleteSnapshotError encountered with the earlier exporter.
revision = previous_config.get("model_revision") if previous_config else None
if revision is None:
    try:
        config_path = Path(hf_hub_download(MODEL_NAME, "config.json", local_files_only=True))
    except Exception as exc:
        print("Config not resolved from cache; checking Hugging Face:", type(exc).__name__)
        config_path = Path(hf_hub_download(MODEL_NAME, "config.json"))
    revision = config_path.parent.name
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Could not determine a pinned HF commit from the cache path.")
else:
    config_path = Path(hf_hub_download(MODEL_NAME, "config.json", revision=revision))
base_config = json.loads(config_path.read_text(encoding="utf-8"))
assert base_config["model_type"] == "gemma2" and base_config["hidden_size"] == 3584
if previous_config:
    experiment = dict(previous_config)  # Preserve the original training metadata during re-evaluation.
else:
    experiment.update({"model_revision": revision, "versions": versions, "python": sys.version,
                       "max_train_hours": MAX_TRAIN_HOURS, "gpu": torch.cuda.get_device_name(0)})
    (RUN_DIR / "run_config.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=revision)
tokenizer.padding_side = "right"
assert all(x is not None for x in [tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id])
assert MAX_LENGTH <= base_config["max_position_embeddings"]

prefix_text = "Choose which response a human would prefer: A, B, or tie.\n\n[Prompt]\n"
a_header_text = "\n\n[Response A]\n"
b_header_text = "\n\n[Response B]\n"
suffix_text = "\n\n[Decision: A, B, or tie]"
omission_text = "\n[... omitted ...]\n"
def encode_text(text):
    return tokenizer.encode(text, add_special_tokens=False)
prefix_ids, a_header_ids, b_header_ids, suffix_ids, omission_ids = [
    encode_text(x) for x in [prefix_text, a_header_text, b_header_text, suffix_text, omission_text]
]
overhead = sum(map(len, [prefix_ids, a_header_ids, b_header_ids, suffix_ids])) + 2
assert MAX_LENGTH - overhead - PROMPT_BUDGET >= 2 * (len(omission_ids) + 2)


def trim_tokens(ids, budget):
    if len(ids) <= budget:
        return list(ids)
    if budget <= len(omission_ids):
        return list(ids[:budget])
    remain = budget - len(omission_ids)
    head, tail = (remain + 1) // 2, remain // 2
    return list(ids[:head]) + omission_ids + (list(ids[-tail:]) if tail else [])


def allocate_budgets(lengths):
    lp, la, lb = map(int, lengths)
    available = MAX_LENGTH - overhead
    if lp + la + lb <= available:
        return lp, la, lb  # The key fix: no unnecessary prompt truncation.
    p = min(lp, PROMPT_BUDGET)
    answer_space = available - p
    a, b = min(la, answer_space // 2), min(lb, answer_space // 2)
    spare = answer_space - a - b
    # An odd spare token may favor A by one token, not an entire response.
    extra_a = min(spare, la - a)
    a += extra_a
    b += min(spare - extra_a, lb - b)
    p += min(available - p - a - b, lp - p)
    assert p + a + b <= available
    return p, a, b


def assemble_input(p, a, b):
    budgets = allocate_budgets([len(p), len(a), len(b)])
    pieces = [trim_tokens(x, k) for x, k in zip([p, a, b], budgets)]
    ids = ([tokenizer.bos_token_id] + prefix_ids + pieces[0] + a_header_ids
           + pieces[1] + b_header_ids + pieces[2] + suffix_ids + [tokenizer.eos_token_id])
    assert len(ids) <= MAX_LENGTH
    return ids, [len(x) > k for x, k in zip([p, a, b], budgets)]

preprocessing = {
    "version": "full-if-fits-v2", "max_length": MAX_LENGTH, "prompt_budget": PROMPT_BUDGET,
    "prefix": prefix_text, "a_header": a_header_text, "b_header": b_header_text,
    "suffix": suffix_text, "omission": omission_text, "padding_side": "right",
    "turn_format": "[Turn {i+1}]\n{text}; blank text is [No text]; join with two newlines",
    "label_names": LABEL_NAMES,
}
(RUN_DIR / "preprocessing.json").write_text(json.dumps(preprocessing, indent=2), encoding="utf-8")

499

In [7]:
def tokenize_frame(frame, swapped=False, description="Tokenize"):
    fields = ["prompt_text", "response_b_text", "response_a_text"] if swapped else ["prompt_text", "response_a_text", "response_b_text"]
    def tokenize_batch(batch):
        parts = [tokenizer(batch[c], add_special_tokens=False, truncation=False)["input_ids"] for c in fields]
        result = {k: [] for k in ["input_ids", "attention_mask", "labels", "length", "truncated_prompt", "truncated_a", "truncated_b"]}
        for p, a, b, label in zip(*parts, batch["labels"]):
            ids, flags = assemble_input(p, a, b)
            result["input_ids"].append(ids)
            result["attention_mask"].append([1] * len(ids))
            result["labels"].append([1, 0, 2][label] if swapped else label)
            result["length"].append(len(ids))
            for name, flag in zip(["truncated_prompt", "truncated_a", "truncated_b"], flags):
                result[name].append(flag)
        return result
    dataset = Dataset.from_pandas(frame, preserve_index=False)
    return dataset.map(tokenize_batch, batched=True, batch_size=64,
                       remove_columns=dataset.column_names, desc=description)

train_tokens = tokenize_frame(train_frame, description="Tokenize training (one orientation per row)")
valid_tokens = tokenize_frame(valid_frame, description="Tokenize original validation")
model_columns = ["input_ids", "attention_mask", "labels"]
train_ds = train_tokens.select_columns(model_columns)
# Length-sorted evaluation saves padding. Selection rows remain selection-only.
selection_order = sorted(map(int, selection_idx), key=lambda i: valid_tokens[i]["length"])
selection_ds = valid_tokens.select(selection_order).select_columns(model_columns)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

rows = []
for name, data in [("train", train_tokens), ("valid", valid_tokens)]:
    lengths = np.asarray(data["length"])
    rows.append({"split": name, "rows": len(data), "mean_tokens": float(lengths.mean()),
                 "max_tokens": int(lengths.max()),
                 **{c + "_fraction": float(np.mean(data[c])) for c in ["truncated_prompt", "truncated_a", "truncated_b"]}})
audit_table = pd.DataFrame(rows)
audit_table.to_csv(RUN_DIR / "tokenization_audit.csv", index=False)
display(audit_table)
example_idx = int(np.argmax(valid_tokens["length"]))
decoded = tokenizer.decode(valid_tokens[example_idx]["input_ids"], skip_special_tokens=False)
(RUN_DIR / "decoded_example.txt").write_text(decoded, encoding="utf-8")
print(decoded[:3500])
# Save exact regression examples for the eventual Kaggle exporter/inference notebook.
regression_indices = sorted(set([0, example_idx, len(valid_frame) // 2]))
regression_examples = []
for i in regression_indices:
    regression_examples.append({"id": str(valid_ids[i]),
        **{c: valid_frame.iloc[i][c] for c in ["prompt_text", "response_a_text", "response_b_text"]},
        "input_ids": valid_tokens[i]["input_ids"]})
(RUN_DIR / "tokenization_cases.json").write_text(json.dumps(regression_examples, ensure_ascii=False), encoding="utf-8")

Tokenize original validation: 100%|██████████| 11496/11496 [00:12<00:00, 906.18 examples/s]


,split,rows,mean_tokens,max_tokens,truncated_prompt_fraction,truncated_a_fraction,truncated_b_fraction
0,train,45981,704.076249,2048,0.016181,0.026163,0.026250
1,valid,11496,691.833594,2048,0.014875,0.026009,0.026009


<bos>Choose which response a human would prefer: A, B, or tie.

[Prompt]
[Turn 1]
Can you please help me write a story i thought of today?

[Turn 2]
Write a fanfiction where Brendon Urie and Dallon Weekes work together on an album "Too Weird To Live, Too Rare To Die" in 2013. Write it in a form of series of e-mails where they exchange song ideas. Make them have disagreements on which direction the album should go musically, with Dallon proposing lyrics and sending demos of his ideas and Brendon acting dismissive towards that. Which in turn leaves Dallon frustrated and Brendon gets annoyed at lack of cooperation. Make the emails sound progressively more passive-aggressive. Make both parties have very differing artistic visons and struggle to compromise. End it with them still struggling to find middle ground and agreeing to meet up. Include exchanging demo files, lyric ideas.

[Turn 3]
That was great! Now write about the meet up that followed at Brendons place. Make them have a big conf

29798

## 3. Load the larger model and adapters

Only adapters and the three-class head are trainable. The head is initialized with small nonzero weights to avoid extremely confident random initial predictions. All optimizer-updated parameters are FP32; the frozen backbone is BF16. The architecture, causal attention, and softcapping are unchanged.

`score.weight MISSING` while loading the base is expected. During a **reload**, the saved trained head must match exactly; that is checked below.

In [8]:
def load_backbone():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, revision=revision, num_labels=3,
        problem_type="single_label_classification", id2label=ID2LABEL, label2id=LABEL2ID,
        pad_token_id=tokenizer.pad_token_id, dtype=torch.bfloat16,
        attn_implementation=ATTENTION_BACKEND,
    )
    model.config.use_cache = False
    model.score.float()
    return model


def create_training_model():
    set_seed(SEED)
    backbone = load_backbone()
    torch.nn.init.normal_(backbone.score.weight, mean=0.0, std=0.001)
    config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_RANK, lora_alpha=LORA_ALPHA,
                        lora_dropout=0.05, target_modules="all-linear",
                        modules_to_save=["score"], bias="none")
    model = get_peft_model(backbone, config)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.to(DEVICE)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    assert any("lora_B" in n for n in trainable) and any("score" in n for n in trainable)
    assert all("lora_" in n or "score" in n for n in trainable)
    model.print_trainable_parameters()
    return model


def compute_loss(outputs, labels, num_items_in_batch=None):
    total = F.cross_entropy(outputs.logits.float(), labels.long(), reduction="sum")
    denominator = labels.numel() if num_items_in_batch is None else num_items_in_batch
    return total / denominator


def compute_metrics(result):
    logits = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    probabilities = softmax(np.asarray(logits, dtype=np.float64), axis=-1)
    assert np.isfinite(probabilities).all()
    return {"log_loss": float(log_loss(result.label_ids, probabilities, labels=[0, 1, 2])),
            "accuracy": float(accuracy_score(result.label_ids, probabilities.argmax(1)))}

# Unequal microbatches: summed normalized losses AND gradients must equal a full batch.
x = torch.randn(7, 3, requires_grad=True)
y = torch.tensor([0, 1, 2, 0, 1, 2, 0])
full = F.cross_entropy(x, y)
accumulated = sum(compute_loss(SimpleNamespace(logits=x[i:i+2]), y[i:i+2], len(y)) for i in range(0, 7, 2))
torch.testing.assert_close(full, accumulated)
torch.testing.assert_close(torch.autograd.grad(full, x, retain_graph=True)[0], torch.autograd.grad(accumulated, x)[0])
del x, y, full, accumulated
print("Cross-entropy accumulation check passed.")

model = None if EVALUATE_RUN else create_training_model()

Cross-entropy accumulation check passed.


Loading weights: 100%|██████████| 464/464 [00:14<00:00, 31.04it/s]
[transformers] Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-9b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 108,046,848 || all params: 9,349,763,584 || trainable%: 1.1556


## 4. Preflight: memory-safe physical batch and measured throughput

This runs forward/backward **without an optimizer step**. Model weights are unchanged. It tests a maximum-length batch, then length-bucketed representative examples. It reserves estimated Adam states plus 2 GiB of headroom; no benchmark can guarantee that every later batch will fit.

The runtime estimate includes representative training throughput and measured validation throughput, but disk, thermal, data-loader, and checkpoint overhead vary. It is not a fixed ETA. The 16-hour training-phase stop is separate and saves/evaluates before stopping. Early stopping may finish much sooner.

In [ ]:
def backward_probe(indices, inspect_gradients=False):
    batch = outputs = loss = None
    model.train()
    model.zero_grad(set_to_none=True)
    try:
        batch = {k: v.to(DEVICE) for k, v in collator([train_ds[int(i)] for i in indices]).items()}
        labels = batch.pop("labels")
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch, use_cache=False)
            loss = compute_loss(outputs, labels)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite preflight loss.")
        loss.backward()
        if inspect_gradients:
            norms = {"adapter": 0.0, "head": 0.0}
            for name, p in model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    if not torch.isfinite(p.grad).all():
                        raise RuntimeError(f"Non-finite gradient: {name}")
                    norms["adapter" if "lora_" in name else "head"] += float(p.grad.norm().item())
            assert norms["adapter"] > 0 and norms["head"] > 0, norms
            return norms
    finally:
        model.zero_grad(set_to_none=True)
        del batch, outputs, loss


@torch.inference_mode()
def inference_probe(indices, batch_size):
    model.eval()
    torch.cuda.synchronize()
    started = time.perf_counter()
    for start in range(0, len(indices), batch_size):
        batch = collator([
            {k: valid_tokens[int(i)][k] for k in ["input_ids", "attention_mask"]}
            for i in indices[start:start+batch_size]
        ])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**batch, use_cache=False).logits
        assert torch.isfinite(logits).all()
    torch.cuda.synchronize()
    return (time.perf_counter() - started) / len(indices)

benchmark = {}
if not EVALUATE_RUN:
    train_sorted = np.argsort(np.asarray(train_tokens["length"]))
    # Same 64 examples and sequence-length distribution for every candidate.
    representative = train_sorted[np.linspace(0, len(train_sorted)-1, 64).astype(int)]
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimizer_reserve = 8 * param_count  # Two FP32 Adam moment tensors; gradients were measured.
    memory_limit = min(total_bytes, free_bytes) - 2 * 2**30
    benchmark_rows = []
    candidates = [int(previous_config["physical_batch_size"])] if RESUME_CHECKPOINT else BATCH_CANDIDATES
    for bs in candidates:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        failure = None
        try:
            backward_probe(train_sorted[-bs:], inspect_gradients=True)
            peak = torch.cuda.max_memory_allocated()
            if peak + optimizer_reserve > memory_limit:
                failure = "insufficient optimizer/headroom reserve"
            else:
                torch.cuda.synchronize()
                started = time.perf_counter()
                for start in range(0, len(representative), bs):
                    backward_probe(representative[start:start+bs])
                torch.cuda.synchronize()
                seconds_per_example = (time.perf_counter()-started) / len(representative)
                benchmark_rows.append({"batch_size": bs, "seconds_per_example": seconds_per_example,
                                       "peak_allocated_gib": peak / 2**30, "status": "ok"})
        except torch.OutOfMemoryError:
            failure = "GPU out of memory in preflight"
        finally:
            model.zero_grad(set_to_none=True)
            gc.collect(); torch.cuda.empty_cache()
        if failure:
            benchmark_rows.append({"batch_size": bs, "seconds_per_example": None,
                                   "peak_allocated_gib": torch.cuda.max_memory_allocated()/2**30,
                                   "status": failure})
            # Larger batches will not rescue a failed memory test.
            break
    if not any(r["status"] == "ok" for r in benchmark_rows) and not RESUME_CHECKPOINT:
        bs = 1
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        backward_probe(train_sorted[-1:], inspect_gradients=True)
        peak = torch.cuda.max_memory_allocated()
        if peak + optimizer_reserve > memory_limit:
            raise RuntimeError("Even batch 1 has insufficient memory headroom. Stop and share the preflight output.")
        torch.cuda.synchronize(); started = time.perf_counter()
        for i in representative:
            backward_probe([i])
        torch.cuda.synchronize()
        benchmark_rows.append({"batch_size": 1, "seconds_per_example": (time.perf_counter()-started)/len(representative),
                               "peak_allocated_gib": peak/2**30, "status": "ok"})
    safe = [r for r in benchmark_rows if r["status"] == "ok"]
    if not safe:
        raise RuntimeError("The saved resume batch no longer fits. Close other GPU sessions; do not silently change resume settings.")
    chosen = min(safe, key=lambda r: r["seconds_per_example"])
    BATCH_SIZE = int(chosen["batch_size"])
    GRAD_ACCUMULATION = EFFECTIVE_BATCH_SIZE // BATCH_SIZE
    # A training-safe physical batch is conservative for inference too.
    EVAL_BATCH_SIZE = BATCH_SIZE
    valid_sorted = np.argsort(np.asarray(valid_tokens["length"]))
    eval_indices = valid_sorted[np.linspace(0, len(valid_sorted)-1, 64).astype(int)]
    eval_seconds = inference_probe(eval_indices, EVAL_BATCH_SIZE)
    train_hours = chosen["seconds_per_example"] * len(train_ds) * MAX_EPOCHS / 3600
    periodic_eval_hours = eval_seconds * len(selection_ds) * math.ceil(MAX_EPOCHS * 4) / 3600
    final_hours = eval_seconds * len(valid_frame) * (2 if RUN_SWAP_TTA else 1) / 3600
    projected_hours = min(1.15*(train_hours + periodic_eval_hours), MAX_TRAIN_HOURS) + 1.15*final_hours
    benchmark = {"candidates": benchmark_rows, "physical_batch_size": BATCH_SIZE,
                 "gradient_accumulation_steps": GRAD_ACCUMULATION,
                 "training_hours_uncapped_estimate": train_hours,
                 "periodic_validation_hours_estimate": periodic_eval_hours,
                 "final_evaluation_hours_estimate": final_hours,
                 "total_compute_hours_with_cap_estimate": projected_hours}
    display(pd.DataFrame(benchmark_rows))
    print(f"Selected batch {BATCH_SIZE} x accumulation {GRAD_ACCUMULATION} = {EFFECTIVE_BATCH_SIZE}")
    print(f"Approximate remaining compute with training cap: {projected_hours:.1f} hours; not a guarantee.")
    if projected_hours > 20:
        print("WARNING: estimated total exceeds 20 hours. The training phase still stops at MAX_TRAIN_HOURS.")
    (RUN_DIR / "benchmark.json").write_text(json.dumps(benchmark, indent=2), encoding="utf-8")
    experiment.update({"physical_batch_size": BATCH_SIZE, "gradient_accumulation_steps": GRAD_ACCUMULATION,
                       "eval_batch_size": EVAL_BATCH_SIZE, "trainable_parameters": param_count,
                       "max_train_hours": MAX_TRAIN_HOURS})
    (RUN_DIR / "run_config.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")
    model.zero_grad(set_to_none=True)
    gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
else:
    experiment = previous_config
    BATCH_SIZE = int(experiment["physical_batch_size"])
    GRAD_ACCUMULATION = int(experiment["gradient_accumulation_steps"])
    EVAL_BATCH_SIZE = int(experiment["eval_batch_size"])
    benchmark_path = RUN_DIR / "benchmark.json"
    if benchmark_path.is_file():
        benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))

## 5. Train, checkpoint, and save the selected adapter

Checkpoints are compared on the **selection subset only**, approximately every quarter epoch. The audit subset is not used by Trainer. Early stopping tolerates two evaluation checks without a meaningful improvement; it is not a guarantee of the globally best epoch.

The time guard is checked after optimizer updates; validation/save work can extend beyond the threshold. On a resumed invocation the 16-hour allowance starts again. A manual `KeyboardInterrupt` keeps the latest previously saved checkpoint, not unsaved updates.

In [ ]:
class RunMonitor(TrainerCallback):
    def __init__(self, output_dir, max_hours):
        self.output_dir = Path(output_dir)
        self.max_seconds = float(max_hours) * 3600
        self.start = None
        self.stopped_for_time = False
        self.last_evaluated_step = -1

    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if time.perf_counter() - self.start >= self.max_seconds:
            self.stopped_for_time = True
            control.should_training_stop = True
            control.should_evaluate = True
            control.should_save = True
        return control

    def on_evaluate(self, args, state, control, **kwargs):
        self.last_evaluated_step = state.global_step

    def on_epoch_end(self, args, state, control, **kwargs):
        # Include the final partial interval without repeating a time-stop evaluation.
        if state.global_step != self.last_evaluated_step:
            control.should_evaluate = True
            control.should_save = True
        return control

    def on_log(self, args, state, control, **kwargs):
        pd.DataFrame(state.log_history).to_csv(self.output_dir / "training_log.csv", index=False)

train_metrics = {}
training_peak = None
if not EVALUATE_RUN:
    steps_per_epoch = math.ceil(math.ceil(len(train_ds) / BATCH_SIZE) / GRAD_ACCUMULATION)
    eval_steps = max(1, math.ceil(steps_per_epoch / 4))
    warmup_steps = max(1, round(0.05 * steps_per_epoch * MAX_EPOCHS))
    argument_names = inspect.signature(TrainingArguments).parameters
    length_grouping = ({"train_sampling_strategy": "group_by_length"}
                       if "train_sampling_strategy" in argument_names else {"group_by_length": True})
    training_args = TrainingArguments(
        output_dir=str(RUN_DIR / "checkpoints"), num_train_epochs=MAX_EPOCHS,
        learning_rate=LEARNING_RATE, lr_scheduler_type="linear", warmup_steps=warmup_steps,
        weight_decay=0.01, max_grad_norm=1.0, optim="adamw_torch",
        bf16=True, fp16=False,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUMULATION,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps", save_strategy="steps", eval_steps=eval_steps, save_steps=eval_steps,
        load_best_model_at_end=True, metric_for_best_model="log_loss", greater_is_better=False,
        save_total_limit=2, logging_steps=25, logging_first_step=True,
        report_to="none", label_names=["labels"], dataloader_num_workers=0,
        dataloader_pin_memory=True, seed=SEED, data_seed=SEED, disable_tqdm=False,
        **length_grouping,
    )
    monitor = RunMonitor(RUN_DIR, MAX_TRAIN_HOURS)
    trainer = Trainer(
        model=model, args=training_args, train_dataset=train_ds, eval_dataset=selection_ds,
        data_collator=collator, processing_class=tokenizer,
        compute_loss_func=compute_loss, compute_metrics=compute_metrics,
        callbacks=[monitor, EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0005)],
    )
    torch.cuda.reset_peak_memory_stats()
    result = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
    training_peak = torch.cuda.max_memory_allocated() / 2**30
    train_metrics = dict(result.metrics)
    best_selection_loss = float(trainer.state.best_metric)
    best_checkpoint = trainer.state.best_model_checkpoint
    ADAPTER_DIR = RUN_DIR / "adapter"
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    trainer.save_state()
    pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR / "training_log.csv", index=False)
    training_summary = {
        "metrics": train_metrics, "best_selection_log_loss": best_selection_loss,
        "best_checkpoint": best_checkpoint, "stopped_for_time_budget": monitor.stopped_for_time,
        "training_peak_allocated_gib": training_peak, "completed_epoch": float(trainer.state.epoch),
    }
    (RUN_DIR / "training_summary.json").write_text(json.dumps(training_summary, indent=2), encoding="utf-8")
    print("Best trained adapter + head:", ADAPTER_DIR)
    print(json.dumps(training_summary, indent=2))
    # Trainer owns model/optimizer references. Release it before loading a fresh copy.
    del trainer, model, result, monitor
    model = None
    gc.collect(); torch.cuda.empty_cache()
else:
    ADAPTER_DIR = RUN_DIR / "adapter"
    training_summary = json.loads((RUN_DIR / "training_summary.json").read_text(encoding="utf-8"))
    train_metrics = training_summary["metrics"]
    best_selection_loss = training_summary["best_selection_log_loss"]
    training_peak = training_summary["training_peak_allocated_gib"]

## 6. Reload the saved head/adapters, then collect original and swapped predictions

Only one model is held on the GPU. We use length-sorted batches and restore original ID order. There is no quantization or merging in this evaluation. Changing dtype or quantization for Kaggle later must be checked against these saved predictions.

Full validation outputs are cached using the adapter hash, tokenizer settings, IDs, and GPU runtime versions. Re-running this section after a successful pass can reuse the file rather than repeat inference. An out-of-memory batch is retried at a smaller physical batch; it is never replaced with dummy predictions.

In [ ]:
adapter_file = ADAPTER_DIR / "adapter_model.safetensors"
if not adapter_file.is_file():
    raise FileNotFoundError(f"Missing saved adapter: {adapter_file}")
adapter_hash = sha256_file(adapter_file)
adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text(encoding="utf-8"))
assert adapter_config["task_type"] == "SEQ_CLS" and "score" in adapter_config["modules_to_save"]
saved_tensors = load_file(str(adapter_file), device="cpu")
head_keys = [k for k in saved_tensors if "score" in k and k.endswith("weight")]
assert len(head_keys) == 1
saved_head = saved_tensors[head_keys[0]].float().clone()
assert tuple(saved_head.shape) == (3, 3584)
del saved_tensors

model = PeftModel.from_pretrained(load_backbone(), str(ADAPTER_DIR), is_trainable=False)
loaded_head = model.get_base_model().score.modules_to_save["default"].weight.detach().cpu().float()
torch.testing.assert_close(loaded_head, saved_head, rtol=0, atol=0)
assert any("lora_B" in n for n, _ in model.named_parameters())
model.requires_grad_(False)
model.gradient_checkpointing_disable()
model.config.use_cache = False
model.to(DEVICE).eval()
del saved_head, loaded_head
print("Reloaded 9B adapter and exact trained classification head.")


@torch.inference_mode()
def predict_logits(data, model, batch_size, description="Predict"):
    lengths = np.asarray(data["length"])
    order = np.argsort(lengths, kind="stable")
    logits = np.empty((len(data), 3), dtype=np.float32)
    pos = 0
    batch_size = int(batch_size)
    progress = tqdm(total=len(data), desc=description)
    started = time.perf_counter()
    try:
        while pos < len(order):
            indices = order[pos:pos + batch_size]
            batch = output = None
            oom = False
            try:
                features = [{k: data[int(i)][k] for k in ["input_ids", "attention_mask"]} for i in indices]
                batch = {k: v.to(DEVICE) for k, v in collator(features).items()}
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    output = model(**batch, use_cache=False).logits
                if output.shape != (len(indices), 3) or not torch.isfinite(output).all():
                    raise RuntimeError("Non-finite or incorrectly shaped inference logits.")
                logits[indices] = output.float().cpu().numpy()
            except torch.OutOfMemoryError:
                if batch_size == 1:
                    raise
                oom = True
            finally:
                del batch, output
            if oom:
                batch_size = max(1, batch_size // 2)
                gc.collect(); torch.cuda.empty_cache()
                print("Retrying with inference batch", batch_size)
                continue
            pos += len(indices)
            progress.update(len(indices))
    finally:
        progress.close()
    return logits, time.perf_counter() - started


def cached_predict(data, orientation):
    signature_data = {"adapter_sha256": adapter_hash, "ids": valid_ids.tolist(),
                      "orientation": orientation, "preprocessing": preprocessing,
                      "versions": versions, "dtype": "bf16-autocast", "batch": EVAL_BATCH_SIZE}
    signature = hashlib.sha256(json.dumps(signature_data, sort_keys=True).encode()).hexdigest()
    path = RUN_DIR / f"valid_9b_{orientation}.npz"
    if path.is_file():
        with np.load(path, allow_pickle=False) as saved:
            if saved["signature"].item() == signature:
                assert np.array_equal(saved["ids"].astype(str), valid_ids)
                cached = saved["logits"].copy()
                assert cached.shape == (len(valid_ids), 3) and np.isfinite(cached).all()
                print("Reusing", path.name)
                return cached, float(saved["seconds"])
    logits, seconds = predict_logits(data, model, EVAL_BATCH_SIZE, f"9B validation: {orientation}")
    np.savez_compressed(path, logits=logits, ids=valid_ids, labels=y_valid,
                        signature=np.asarray(signature), seconds=np.asarray(seconds))
    return logits, seconds

new_logits, original_seconds = cached_predict(valid_tokens, "original")
reloaded_selection_loss = float(log_loss(y_valid[selection_idx], softmax(new_logits[selection_idx].astype(np.float64), axis=-1), labels=[0, 1, 2]))
reload_difference = reloaded_selection_loss - best_selection_loss
print(f"Selection reload log-loss difference: {reload_difference:+.6f}")
if abs(reload_difference) > 0.005:
    raise RuntimeError("Reload differs materially from selected checkpoint. Stop and share the outputs before using this model.")

swapped_logits = None
swapped_seconds = 0.0
if RUN_SWAP_TTA:
    swapped_tokens = tokenize_frame(valid_frame, swapped=True, description="Tokenize swapped validation")
    swapped_logits, swapped_seconds = cached_predict(swapped_tokens, "swapped")
    del swapped_tokens
# Save raw logits: swapped column 0 means ORIGINAL B wins; do not forget remapping.
np.savez_compressed(RUN_DIR / "valid_logits.npz", ids=valid_ids, labels=y_valid,
                    logits=new_logits, swapped_logits=swapped_logits if swapped_logits is not None else np.empty((0, 3)))
# No more GPU work is needed for model/temperature/ensemble comparisons.
del model
model = None
gc.collect(); torch.cuda.empty_cache()

## 7. Choose temperature, swap-TTA, and a 2B/9B probability blend

Temperatures and weights are fitted on **selection only**. Swapped outputs are mapped back using `[1, 0, 2]`, then probabilities (not class IDs) are averaged. The 2B model is included only when the small search supports it; pure 9B and pure 2B are candidates too.

Among recipes within 0.0005 selection log loss of the best, prefer fewer passes, then no ensemble, then less temperature adjustment. This avoids buying an expensive extra inference pass for a negligible selection gain. The estimated pass cost uses a rough relative 2B cost of 0.3; it is not a measured Kaggle speed ratio.

**Important:** the audit columns are computed only after the recipe is frozen. They diagnose whether the gain generalizes; do not repeatedly tune against them. Full-fold numbers are descriptive and are not independent scores after selection.

In [ ]:
def checked_softmax(logits, temperature=1.0):
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be positive and finite.")
    p = softmax(np.asarray(logits, dtype=np.float64) / temperature, axis=-1)
    if not np.isfinite(p).all():
        raise ValueError("Invalid probabilities.")
    return p


def candidate_probs(z9, zswap, z2, t9, t2, use_tta, old_weight):
    if not 0 <= old_weight <= 1:
        raise ValueError("old_weight must be between zero and one.")
    if old_weight == 1:
        return checked_softmax(z2, t2)
    p9 = checked_softmax(z9, t9)
    if use_tta:
        if zswap is None:
            raise ValueError("Swapped predictions are required for TTA.")
        p9 = 0.5 * (p9 + checked_softmax(zswap, t9)[:, [1, 0, 2]])
    if old_weight == 0:
        return p9
    return (1.0-old_weight) * p9 + old_weight * checked_softmax(z2, t2)


def fit_recipe(z9, zswap, z2, labels, temperatures, old_weights, tolerance):
    # This function receives SELECTION arrays only. It cannot access audit labels.
    t2_rows = [(float(log_loss(labels, checked_softmax(z2, t), labels=[0, 1, 2])), t) for t in temperatures]
    t2 = min(t2_rows, key=lambda pair: (pair[0], abs(pair[1]-1)))[1]
    search = []
    for use_tta in ([False, True] if zswap is not None else [False]):
        for t9 in temperatures:
            for w in old_weights:
                probs = candidate_probs(z9, zswap, z2, t9, t2, use_tta, w)
                search.append({"temperature_9b": float(t9), "temperature_2b": float(t2),
                    "use_9b_tta": bool(use_tta), "weight_2b": float(w),
                    "selection_log_loss": float(log_loss(labels, probs, labels=[0, 1, 2])),
                    "relative_pass_cost_proxy": float(1 + int(use_tta) + (0.3 if w > 0 else 0))})
    # Keep a cheap calibrated 2B-only fallback rather than force a worse new model.
    search.append({"temperature_9b": 1.0, "temperature_2b": float(t2), "use_9b_tta": False,
                   "weight_2b": 1.0, "selection_log_loss": min(t2_rows)[0],
                   "relative_pass_cost_proxy": 0.3})
    best_loss = min(r["selection_log_loss"] for r in search)
    eligible = [r for r in search if r["selection_log_loss"] <= best_loss + tolerance]
    chosen = min(eligible, key=lambda r: (
        r["relative_pass_cost_proxy"], int(0 < r["weight_2b"] < 1),
        abs(r["temperature_9b"]-1) + abs(r["temperature_2b"]-1), r["selection_log_loss"]))
    return dict(chosen), pd.DataFrame(search).sort_values("selection_log_loss").reset_index(drop=True)

recipe, recipe_search = fit_recipe(
    new_logits[selection_idx], None if swapped_logits is None else swapped_logits[selection_idx],
    old_logits[selection_idx], y_valid[selection_idx], TEMPERATURES, OLD_MODEL_WEIGHTS, SELECTION_TOLERANCE,
)
recipe_search.to_csv(RUN_DIR / "postprocess_search.csv", index=False)
recipe.update({
    "weight_9b": 1.0 - recipe["weight_2b"], "selection_tolerance": SELECTION_TOLERANCE,
    "formula": "(1-w2)*mean(softmax(z9/T9), remapped softmax(z9_swapped/T9) if TTA) + w2*softmax(z2/T2)",
    "model_9b": MODEL_NAME, "adapter_9b": str(ADAPTER_DIR), "adapter_9b_sha256": adapter_hash,
    "model_2b": "google/gemma-2-2b-it", "baseline_run": str(BASELINE_RUN),
    "2b_preprocessing": "EXACT original 1536-token / prompt-budget-256 implementation; do not replace with v2",
    "9b_preprocessing": preprocessing,
    "selection_policy": "selection subset only; prefer cheaper candidates within tolerance; audit not used",
    "kaggle_ready": False,
    "deployment_note": "This recipe needs a newly validated offline 9B inference/export path. Do not reuse the 2B-only notebook unchanged.",
})
(RUN_DIR / "postprocess_recipe.json").write_text(json.dumps(recipe, indent=2), encoding="utf-8")
# Only now evaluate the frozen recipe on all validation rows (including audit).
selected_probs = candidate_probs(new_logits, swapped_logits, old_logits,
    recipe["temperature_9b"], recipe["temperature_2b"], recipe["use_9b_tta"], recipe["weight_2b"])
raw_new = checked_softmax(new_logits)
models = {"2B original (previous run)": old_probabilities,
          "2B temperature only": checked_softmax(old_logits, recipe["temperature_2b"]),
          "9B original, T=1": raw_new}
if swapped_logits is not None:
    models["9B A/B TTA, T=1"] = 0.5*(raw_new + checked_softmax(swapped_logits)[:, [1, 0, 2]])
models["Selected recipe"] = selected_probs
comparison_rows = []
for name, probs in models.items():
    comparison_rows.append({"model": name,
        "selection_log_loss": float(log_loss(y_valid[selection_idx], probs[selection_idx], labels=[0, 1, 2])),
        "audit_log_loss": float(log_loss(y_valid[audit_idx], probs[audit_idx], labels=[0, 1, 2])),
        "full_validation_log_loss": float(log_loss(y_valid, probs, labels=[0, 1, 2])),
        "full_validation_accuracy": float(accuracy_score(y_valid, probs.argmax(1)))})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RUN_DIR / "ensemble_comparison.csv", index=False)
display(comparison)
print("Selected recipe:", {k: recipe[k] for k in ["weight_9b", "weight_2b", "temperature_9b", "temperature_2b", "use_9b_tta"]})
print("Note: the full-fold score is descriptive after checkpoint and recipe selection.")

## 8. Save the result package and diagnostics

The small ZIP is for reporting back here; it contains **no model weights, source conversations, or secrets**. Preserve the full run directory locally for the future exporter. `valid_logits.npz` permits later calibration/ensemble analysis without retraining.

The old 2B model alone is already deployable with your working Kaggle notebook. A 9B model is a different deployment workload: FP32 does not fit in two 16 GB T4s just by pooling their memory. We must validate reduced-precision/quantized inference or an appropriate accelerator, and measure the nine-hour Kaggle deadline before submitting the larger model or multiple passes. No claim about T4 runtime is made by this local benchmark.

In [ ]:
assert selected_probs.shape == (len(valid_ids), 3)
assert np.isfinite(selected_probs).all() and np.allclose(selected_probs.sum(1), 1, atol=1e-8)
valid_output = pd.DataFrame({"id": valid_ids, "label": y_valid,
    "subset": inner_manifest["subset"], "length": valid_tokens["length"],
    "num_turns": valid_frame["num_turns"],
    **{c: valid_tokens[c] for c in ["truncated_prompt", "truncated_a", "truncated_b"]}})
for j, c in enumerate(LABEL_COLUMNS):
    valid_output[c] = selected_probs[:, j]
valid_output["row_log_loss"] = -np.log(np.clip(selected_probs[np.arange(len(y_valid)), y_valid], 1e-15, 1))
valid_output.to_csv(RUN_DIR / "valid_predictions.csv", index=False)

slices = []
for name, mask in {
    "all": np.ones(len(y_valid), dtype=bool),
    "any_truncation": valid_output[["truncated_prompt", "truncated_a", "truncated_b"]].any(axis=1).to_numpy(),
    "multi_turn": (valid_output.num_turns > 1).to_numpy(),
    "tie": y_valid == 2,
}.items():
    mask = mask & np.isin(np.arange(len(y_valid)), audit_idx)
    if mask.any():
        slices.append({"slice": name, "audit_rows": int(mask.sum()),
            "old_2b_log_loss": float(log_loss(y_valid[mask], old_probabilities[mask], labels=[0, 1, 2])),
            "selected_log_loss": float(log_loss(y_valid[mask], selected_probs[mask], labels=[0, 1, 2]))})
pd.DataFrame(slices).to_csv(RUN_DIR / "audit_slices.csv", index=False)

summary = {
    "run_directory": str(RUN_DIR), "config": experiment,
    "public_baseline_score_from_user": 0.97220,
    "previous_2b_validation_log_loss": old_loss,
    "best_checkpoint_selection_log_loss": best_selection_loss,
    "reload_selection_log_loss_difference": reload_difference,
    "training": training_summary, "benchmark": benchmark,
    "validation_comparison": comparison.to_dict(orient="records"),
    "selected_recipe": recipe,
    "inference_seconds": {"9b_original": original_seconds, "9b_swapped": swapped_seconds},
    "tokenization_audit": audit_table.to_dict(orient="records"),
    "rank_prediction": None,
}
(RUN_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
# Copy only small diagnostic outputs into a ZIP, not model weights or private text.
import zipfile
report_names = ["run_summary.json", "run_config.json", "training_log.csv", "training_summary.json",
                "benchmark.json", "ensemble_comparison.csv", "postprocess_search.csv",
                "postprocess_recipe.json", "audit_slices.csv", "tokenization_audit.csv"]
report_path = RUN_DIR / "results_for_review.zip"
with zipfile.ZipFile(report_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in report_names:
        path = RUN_DIR / name
        if path.is_file():
            z.write(path, arcname=name)
print("FINISHED. Upload this small report ZIP:", report_path)
print("Keep the adapter, split, tokenizer, and valid_logits.npz in:", RUN_DIR)
if comparison.iloc[-1]["audit_log_loss"] >= comparison.iloc[0]["audit_log_loss"]:
    print("CAUTION: the selected recipe did not beat the original 2B model on the audit subset.")

## Research basis and limits

The 2024 LMSYS competition is historical evidence for this task, not a verified explanation of every present-day entrant on the rolling Getting Started leaderboard.

- [1st-place write-up](https://www.kaggle.com/competitions/lmsys-chatbot-arena/discussion/527629): Gemma 2 9B LoRA on linear layers, larger-model distillation, and TTA. We borrow the model/training direction, **not** its large teacher models or published trained weights.
- [2nd-place write-up](https://www.kaggle.com/competitions/lmsys-chatbot-arena/discussion/527685): prompt-group validation, swapping, ensembles, and efficient batching. Its full-parameter multi-A100 training is **not** reproduced here. Our 50% deterministic swap is cheaper than its paired full-swap method.
- [PEFT LoRA](https://huggingface.co/docs/peft/package_reference/lora): adapter targets and saving the classification head.
- [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer): custom accumulated loss, checkpoint selection, callbacks, and gradient checkpointing.
- [Gemma 2 implementation](https://github.com/huggingface/transformers/blob/main/src/transformers/models/gemma2/modeling_gemma2.py): preserve the architecture's attention behavior.
- [Temperature scaling](https://arxiv.org/abs/1706.04599): probability calibration, tested rather than assumed to help.
- [Competition requirements](https://www.kaggle.com/competitions/llm-classification-finetuning): offline notebook submissions and the nine-hour runtime limit remain a separate deployment constraint.

The chosen hyperparameters, token allocation fix, runtime guard, and inexpensive recipe search are this experiment's design choices, not settings claimed to guarantee a medal or a percentile. A larger model plus an ensemble is a plausible route to a better score; only validation and an actual submission will establish the result.